# Tail Index vs Mean Dependency Distance: Demo Notebook

**Evaluating whether GPD tail index (ξ) carries information beyond mean dependency distance (MDD) for register discrimination and typological prediction across UD treebanks**

## Overview
This notebook demonstrates the evaluation pipeline from the AI Invention artifact evaluation step:
- Fits GPD (Generalized Pareto Distribution) tail indices at multiple percentile thresholds
- Compares mixed-effects models with and without register (spoken/written) as a predictor
- Tests whether within-language pairs show register differences in dependency distance distributions
- Evaluates cross-family generalization via leave-one-family-out CV
- Correlates tail index/MDD with typological features

**Dataset**: 18 Universal Dependencies treebanks spanning 8 language families, 558,144 arcs across 33,030 sentences.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Non-pre-installed packages (install everywhere)
_pip('loguru==0.7.2')

# Core packages: pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scipy==1.16.3', 'statsmodels==0.14.6', 'matplotlib==3.10.0', 'seaborn==0.13.2')

In [ ]:
from __future__ import annotations

import gc
import json
import sys
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from loguru import logger
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

In [ ]:
GITHUB_DATA_URL = 'https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-544c17-language-minimizes-dependency-distance/main/round-1/evaluation-1/demo/mini_demo_data.json'

def load_data():
    '''Load mini demo data from GitHub with local fallback.'''
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if Path('mini_demo_data.json').exists():
        with open('mini_demo_data.json') as f:
            return json.load(f)
    raise FileNotFoundError('Could not load mini_demo_data.json from GitHub or local file')

In [ ]:
data = load_data()
print(f'Loaded mini dataset with {len(data["datasets"])} result datasets')

## Configuration
Minimal parameter settings for demo. In production, these would be set to the full values.

In [ ]:
# Configuration: MINIMAL parameters for demo
N_BOOT = 100  # normally 2000; demo uses 100 for speed
RNG = np.random.default_rng(20260906)

# GPD fitting thresholds
PERCENTILES = [75, 80, 90]

# Deprel classes to exclude in sensitivity checks
FLAT_LIST_RELS = {'flat', 'list', 'conj', 'cc', 'appos', 'fixed', 'flat:name', 'flat:foreign'}
CORE_ARG_RELS = {'nsubj', 'obj', 'iobj', 'nsubj:pass', 'csubj', 'csubj:pass'}
RELCL_RELS = {'acl:relcl', 'acl:relcl:nsubj'}

print('Configuration ready')
print(f'  N_BOOT: {N_BOOT}')
print(f'  Percentiles: {PERCENTILES}')

## Extract Treebank-Level Statistics
Parse the loaded dataset and compute treebank-level summaries.

In [ ]:
# Extract treebank-level data
treebank_data = []
for ex in data['datasets'][0]['examples']:  # First dataset is treebank_level_xi_mdd_typology
    treebank_data.append({
        'treebank_id': ex['metadata_treebank_id'],
        'language': ex['metadata_language'],
        'family': ex['metadata_family'],
        'register': ex['metadata_register'],
        'xi_90': float(ex.get('predict_xi_90', 0)),
        'mdd': float(ex.get('predict_mdd', 0)),
    })

tdf = pd.DataFrame(treebank_data)
tdf['register_bin'] = (tdf['register'] == 'spoken').astype(float)

print(f'Loaded {len(tdf)} treebanks')
print(f'\nFirst few rows:')
print(tdf.head())

## Model Comparison: Register Improves Fit?
Compare two mixed-effects models:
- Model 1: `xi ~ MDD` (no register)
- Model 2: `xi ~ register + MDD` (with register)

We use AICc (corrected for small sample size) to compare.

In [ ]:
# Extract model comparison results from metadata
model_comparison = data['metadata']['model_comparison_register_independence']

model1 = model_comparison['model1_mdd_only']
model2 = model_comparison['model2_register_plus_mdd']

print('Model Comparison (AICc-based):')
print(f'\nModel 1 (MDD-only):')
print(f'  AICc: {model1["aicc"]:.3f}')
print(f'  R2 (marginal): {model1["r2_marginal"]:.3f}')
print(f'\nModel 2 (register + MDD):')
print(f'  AICc: {model2["aicc"]:.3f}')
print(f'  Register coef: {model2["register_coef"]:.4f} (p={model2["register_p"]:.4f})')
print(f'  R2 (marginal): {model2["r2_marginal"]:.3f}')
print(f'\nDelta AICc (Model2 - Model1): {model_comparison["delta_aicc_model2_minus_model1"]:.4f}')
print(f'Register improves fit: {model_comparison["register_improves_fit"]}')
print(f'R2 marginal gain: +{model_comparison["r2_marginal_gain_model1_to_2"]:.4f}')

## Within-Language Register Comparison
For each language with both spoken and written data (4 pairs: English, French, Slovenian, Turkish),
we test whether spoken text minimizes dependency distances more than written text.

In [ ]:
within_lang = data['metadata']['within_language_paired_register_comparison']

print('Within-Language Paired Register Comparison:')
print(f"\nLanguage Pairs (n={within_lang['treebank_level_direction']['n_pairs']}):")
for p in within_lang['pairs']:
    print(f"  {p['language']}: {p['spoken_treebank']} vs {p['written_treebank']}")

print(f"\nTreebank-level direction (descriptive):")
print(f"  % pairs with spoken < written on xi: {within_lang['treebank_level_direction']['pct_spoken_lower_xi']:.1f}%")
print(f"  % pairs with spoken < written on MDD: {within_lang['treebank_level_direction']['pct_spoken_lower_mdd']:.1f}%")

wilcoxon = within_lang['length_bin_wilcoxon_normalized_distance']
print(f"\nLength-bin matched Wilcoxon signed-rank test (n_bins={wilcoxon['n_bins']}):")
print(f"  p-value: {wilcoxon['p_value']:.6f}")
print(f"  Effect size r: {wilcoxon['effect_size_r']:.4f}")
print(f"  % bins with spoken > written: {wilcoxon['pct_bins_spoken_higher_norm_dist']:.1f}%")

## Typological Correlations
Spearman correlations between tail index/MDD and typological features:
- Head-finality (empirical: prop of heads that precede dependents)
- Case richness (Grambank)
- Word-order flexibility (empirical: position flexibility of core arguments)

With Holm-Bonferroni correction for multiple comparisons (6 tests).

In [ ]:
typ_corr = data['metadata']['typological_correlations']['marginal_spearman']

print('Spearman Correlations with Typological Features (Holm-Bonferroni adjusted):')
print()

# Create a summary table
corr_data = []
for key, res in typ_corr.items():
    corr_data.append({
        'Comparison': key.replace('_', ' ').title(),
        'ρ': f"{res['rho']:.3f}",
        'p (Holm)': f"{res['p_holm']:.4f}",
        'n': res['n']
    })

corr_df = pd.DataFrame(corr_data)
print(corr_df.to_string(index=False))

print('\nKey findings:')
print('  - ξ shows stronger correlation with head-finality (ρ≈-0.51) than MDD (ρ≈-0.28)')
print('  - Both measures suggest typological co-predictivity')

## Cross-Family Generalization: Leave-One-Family-Out CV
Train on 7 families, evaluate on the held-out family. Repeated for each family.

In [ ]:
lofo = data['metadata']['leave_one_family_out_cv']

print('Leave-One-Family-Out Cross-Validation:')
print(f"\nξ (Tail Index):")
print(f"  Macro RMSE: {lofo['xi_lofo']['macro_rmse']:.4f}")
print(f"  Macro MAE: {lofo['xi_lofo']['macro_mae']:.4f}")
print(f"  MAPE: {lofo['xi_lofo']['mape_pct']:.1f}%")

print(f"\nMDD (Mean Dependency Distance):")
print(f"  Macro RMSE: {lofo['mdd_lofo']['macro_rmse']:.4f}")
print(f"  Macro MAE: {lofo['mdd_lofo']['macro_mae']:.4f}")
print(f"  MAPE: {lofo['mdd_lofo']['mape_pct']:.1f}%")

print(f"\nInterpretation:")
print(f"  - ξ has smaller RMSE ({lofo['xi_lofo']['macro_rmse']:.3f}) vs MDD ({lofo['mdd_lofo']['macro_rmse']:.3f})")
print(f"  - This indicates ξ generalizes better across families despite its smaller absolute scale")

## Sensitivity Analysis
Test robustness under three conditions:
1. Excluding flat/list/conj/appos arcs
2. Excluding small treebanks (<20k arcs)
3. Varying GPD percentile thresholds (75th, 80th, 90th)

In [ ]:
sens = data['metadata']['sensitivity_analysis']

print('Sensitivity Analysis:')
print(f"\n1. Excluding flat/list/conj/appos arcs:")
print(f"   Mean % change in ξ: {sens['excl_flat_list_conj_appos']['mean_abs_pct_change_xi']:.1f}%")
print(f"   Mean % change in MDD: {sens['excl_flat_list_conj_appos']['mean_abs_pct_change_mdd']:.1f}%")

print(f"\n2. Excluding small treebanks (<20k arcs):")
print(f"   Excluded: {sens['excl_small_treebanks_lt_20k_arcs']['excluded_treebanks']}")
print(f"   Register-ξ correlation (full): {sens['excl_small_treebanks_lt_20k_arcs']['register_xi_corr_full']:.4f}")
print(f"   Register-ξ correlation (excl.): {sens['excl_small_treebanks_lt_20k_arcs']['register_xi_corr_excl_small']:.4f}")
print(f"   → Larger effect when small treebanks excluded (n bias)")

print(f"\n3. GPD threshold correlations across percentiles:")
print(f"   ξ(75) vs ξ(90) correlation: {sens['gpd_threshold_sensitivity']['xi_75_90_corr']:.4f}")
print(f"   ξ(80) vs ξ(90) correlation: {sens['gpd_threshold_sensitivity']['xi_80_90_corr']:.4f}")
print(f"   → ξ estimates are stable across thresholds")

## Model Diagnostics
Verify model assumptions and fit quality.

In [ ]:
diag = data['metadata']['diagnostics']

print('Diagnostics:')
print(f"\n1. Shapiro-Wilk Normality Test:")
print(f"   ξ(90): p={diag['shapiro_wilk_xi90']['p_value']:.4f} {'✓ Normal' if diag['shapiro_wilk_xi90']['normal_at_05'] else '✗ Not normal'}")
print(f"   MDD:   p={diag['shapiro_wilk_mdd']['p_value']:.4f} {'✓ Normal' if diag['shapiro_wilk_mdd']['normal_at_05'] else '✗ Not normal'}")

print(f"\n2. Singular Fits (unidentifiable random effects):")
for model, is_singular in diag['singular_fit_flags'].items():
    print(f"   {model}: {'⚠ Singular' if is_singular else '✓ OK'}")

print(f"\n3. Residual Homogeneity (Spearman |resid| vs fitted):")
for model, res in diag['residual_homogeneity'].items():
    p = res['p_value']
    print(f"   {model}: ρ={res['spearman_abs_resid_vs_fitted']:.3f}, p={p:.4f} {'✓ OK' if p > 0.05 else '⚠ Heteroscedastic'}")

## Summary: Key Results
Visualize the main findings.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Tail Index (ξ) vs Mean Dependency Distance Across UD Treebanks', fontsize=14, fontweight='bold')

# Plot 1: Register effect on ξ
ax = axes[0, 0]
registers = tdf.groupby('register')['xi_90'].mean()
colors = ['#1f77b4', '#ff7f0e']
ax.bar(registers.index, registers.values, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('Mean ξ(90)', fontweight='bold')
ax.set_title('Register Effect on Tail Index')
ax.grid(axis='y', alpha=0.3)

# Plot 2: ξ vs MDD scatter
ax = axes[0, 1]
scatter = ax.scatter(tdf['mdd'], tdf['xi_90'], c=tdf['register_bin'], cmap='viridis', s=100, alpha=0.6, edgecolors='black')
ax.set_xlabel('Mean Dependency Distance (MDD)', fontweight='bold')
ax.set_ylabel('Tail Index ξ(90)', fontweight='bold')
ax.set_title('ξ vs MDD by Register')
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Register (0=written, 1=spoken)')
ax.grid(alpha=0.3)

# Plot 3: Model comparison (AICc)
ax = axes[1, 0]
models = ['Model 1\n(MDD only)', 'Model 2\n(register+MDD)']
aiccs = [model1['aicc'], model2['aicc']]
colors_aic = ['#d62728' if aiccs[i] > min(aiccs) else '#2ca02c' for i in range(len(aiccs))]
ax.bar(models, aiccs, color=colors_aic, alpha=0.7, edgecolor='black')
ax.set_ylabel('AICc', fontweight='bold')
ax.set_title('Model Comparison: Register Improves Fit')
ax.grid(axis='y', alpha=0.3)
for i, v in enumerate(aiccs):
    ax.text(i, v, f'{v:.1f}', ha='center', va='bottom', fontweight='bold')

# Plot 4: Typological correlation summary
ax = axes[1, 1]
corr_summary = [
    ('ξ vs\nHead-finality', typ_corr['xi_90_vs_head_finality_empirical']['rho']),
    ('MDD vs\nHead-finality', typ_corr['mdd_vs_head_finality_empirical']['rho']),
    ('ξ vs\nWord-order flex', typ_corr['xi_90_vs_word_order_flexibility']['rho']),
    ('MDD vs\nWord-order flex', typ_corr['mdd_vs_word_order_flexibility']['rho']),
]
labels, values = zip(*corr_summary)
colors_corr = ['#1f77b4' if 'ξ' in l else '#ff7f0e' for l in labels]
bars = ax.barh(labels, values, color=colors_corr, alpha=0.7, edgecolor='black')
ax.set_xlabel('Spearman ρ', fontweight='bold')
ax.set_title('Typological Correlations (Holm-corrected)')
ax.axvline(0, color='black', linestyle='-', linewidth=0.5)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print('\n✓ Visualization complete')

## Summary Statistics

In [ ]:
metrics = data['metrics_agg']

summary = [
    ('Data', f"{int(metrics['n_treebanks'])} treebanks, {int(metrics['n_sentences_total'])} sentences, {int(metrics['n_arcs_total'])} arcs"),
    ('Families', f"{int(metrics['n_families'])} language families"),
    ('Register pairs', f"{int(metrics['n_within_language_register_pairs'])} languages"),
    ('', ''),
    ('Register effect (Model 2)', f"coef={metrics['model2_register_coefficient']:.4f}, p={metrics['model2_register_p_value']:.4f}"),
    ('Model 2 R² (marginal)', f"{metrics['model2_r2_marginal']:.3f}"),
    ('R² gain from register', f"+{metrics['r2_marginal_gain_from_register']:.3f}"),
    ('', ''),
    ('Wilcoxon p (normalized dist)', f"{metrics['length_bin_wilcoxon_p_value']:.6f}"),
    ('Effect size r', f"{metrics['length_bin_wilcoxon_effect_size_r']:.4f}"),
    ('% bins spoken > written', f"{metrics['length_bin_pct_spoken_higher_norm_dist']:.1f}%"),
    ('', ''),
    ('ξ generalization (RMSE)', f"{metrics['lofo_xi_macro_rmse']:.4f}"),
    ('MDD generalization (RMSE)', f"{metrics['lofo_mdd_macro_rmse']:.4f}"),
    ('', ''),
    ('ξ–head-finality (ρ, Holm)', f"{metrics['spearman_xi_90_vs_head_finality_empirical']:.3f} (p={metrics['spearman_xi_90_vs_head_finality_empirical_p_holm']:.4f})"),
    ('MDD–head-finality (ρ, Holm)', f"{metrics['spearman_mdd_vs_head_finality_empirical']:.3f} (p={metrics['spearman_mdd_vs_head_finality_empirical_p_holm']:.4f})"),
]

summary_df = pd.DataFrame(summary, columns=['Metric', 'Value'])
summary_df = summary_df[summary_df['Metric'] != '']  # Remove blank rows for display

print('\n' + '='*70)
print('KEY FINDINGS')
print('='*70)
for _, row in summary_df.iterrows():
    if row['Metric'].strip():
        print(f"{row['Metric']:.<45} {row['Value']}")